In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive


'2023101096 (1).py'
 2023101096.py
 AGNEWS_QUANTIZED_20251007_122700
 AGNEWS_QUANTIZED_20251007_122722
 AGNEWS_QUANTIZED_20251007_122734
 AGNEWS_QUANTIZED_20251007_122817
 AGNEWS_QUANTIZED_20251007_122830
 AGNEWS_QUANTIZED_20251007_123415
 AGNEWS_QUANTIZED_20251007_131442
 AGNEWS_QUANTIZED_20251007_132209
 AGNEWS_QUANTIZED_20251007_132656
 AGNEWS_QUANTIZED_20251007_132743
 AGNEWS_QUANTIZED_SAFE_20251007_132944
 AGNEWS_RESULTS_20251005_102908
 colab_datasets
'Colab Notebooks'
 Cover.gdoc
 dsa_ass4.gdoc
 EUbookshop.zip
'Exploring the influence of social media on sustainable fashion consumption: A systematic literature review and future research agenda.gdoc'
 fin.mp4
'Google AI Studio'
 gpt2_agnews_checkpoints
 gpt2_agnews_checkpoints1
 Letter.gdoc
'Peer Review Form R1.docx'
'Peer Review Form-R2.gdoc'
'SaiAditya_Ramanathan_2023101096_2025 (1).gdoc'
 SaiAditya_Ramanathan_2023101096_2025.gdoc
 SaiAditya_Ramanathan_2023101096_2025.pdf
'Saved from Chrome'
'UHV-II Lecture Summaries.gdoc'
'Unti

In [3]:
# Cell 1 — imports & simple utils
import os, time, datetime, glob, json, shutil
import torch
import numpy as np
from copy import deepcopy
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from torch import nn
import psutil
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def cpu_memory_mb():
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / (1024**2)

def print_mem():
    print(f"CPU RAM (MB): {cpu_memory_mb():.1f}")
    if torch.cuda.is_available():
        print(f"CUDA allocated (MB): {torch.cuda.memory_allocated()/1024**2:.1f}")

print("Imports ok. Torch:", torch.__version__)


Imports ok. Torch: 2.8.0+cu126


In [4]:
# Cell 4 — dataset + tokenizer + collator (only needed for evaluation)
MODEL_NAME = 'gpt2'
MAX_LENGTH = 128

print("Loading AG News dataset (this may take a minute)...")
raw = load_dataset('ag_news')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess_batch(examples):
    texts = examples['text']
    enc = tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_LENGTH)
    enc['labels'] = examples['label']
    return enc

train = raw['train'].map(preprocess_batch, batched=True, remove_columns=raw['train'].column_names)
test = raw['test'].map(preprocess_batch, batched=True, remove_columns=raw['test'].column_names)

collator = DataCollatorWithPadding(tokenizer)
print("Tokenization done. Train/test sizes:", len(train), len(test))
print_mem()


Loading AG News dataset (this may take a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Tokenization done. Train/test sizes: 120000 7600
CPU RAM (MB): 1575.5
CUDA allocated (MB): 0.0


In [5]:
# Cell 5 — model class (must match training-time architecture)
class GPT2forClassification(nn.Module):
    def __init__(self, model_name='gpt2', num_labels=4):
        super().__init__()
        # Load base GPT-2 LM model (weights will be reloaded later from state_dict)
        self.gpt = AutoModelForCausalLM.from_pretrained(model_name, return_dict=True)
        # Expose config at top-level so downstream code expecting trainer.model.config works
        self.config = self.gpt.config
        hidden_size = self.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        # Use .model to access base transformer module (GPT2LMHeadModel has .transformer but many HF model wrappers vary)
        # For AutoModelForCausalLM the underlying module is .transformer or .model depending on HF version; we try multiple fallbacks.
        try:
            base_out = self.gpt.model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        except Exception:
            # fallback for older/newer HF variants
            base_out = self.gpt.transformer(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        last_hidden = base_out.last_hidden_state  # (B, L, H)
        seq_lens = attention_mask.sum(dim=1) - 1
        batch_idxs = torch.arange(last_hidden.size(0), device=last_hidden.device)
        pooled = last_hidden[batch_idxs, seq_lens]  # (B, H)
        logits = self.classifier(pooled)
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {'loss': loss, 'logits': logits}
        return {'logits': logits}

# Instantiate model (this creates model object and downloads config/init weights)
model = GPT2forClassification(MODEL_NAME, num_labels=4)
print("Model created. Param count:", sum(p.numel() for p in model.parameters()))
print_mem()


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model created. Param count: 124442884
CPU RAM (MB): 1640.9
CUDA allocated (MB): 0.0


In [6]:
# Cell 6 — minimal Trainer so 'trainer.model' exists in globals exactly as downstream expects
output_dir = "/content/drive/MyDrive/gpt2_agnews_checkpoints"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    logging_steps=50,
    save_steps=1000,
    report_to="none",
)

# Create HF Trainer wrapper (not necessary to run training; it provides .model)
hf_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train if 'train' in globals() else None,
    eval_dataset=test if 'test' in globals() else None,
    data_collator=collator if 'collator' in globals() else None,
)
# Expose name `trainer` as code expects it
trainer = hf_trainer
print("Trainer created. trainer.model class:", trainer.model.__class__)
print("trainer.model.config:", type(trainer.model.config))


Trainer created. trainer.model class: <class '__main__.GPT2forClassification'>
trainer.model.config: <class 'transformers.models.gpt2.configuration_gpt2.GPT2Config'>


In [7]:
# ONE-CELL: suffix-based remapper + safe load (paste & run)
import os, torch, traceback
from collections import OrderedDict

# (Optional) override path here if needed:
# STATE_DICT_PATH = "/content/drive/MyDrive/gpt2_agnews_checkpoints1/final_backup_auto/model_state_dict_cpu.pth"

assert os.path.exists(STATE_DICT_PATH), f"Checkpoint not found: {STATE_DICT_PATH}"
print("STATE_DICT_PATH:", STATE_DICT_PATH)

loaded = torch.load(STATE_DICT_PATH, map_location="cpu")
print("Loaded type:", type(loaded))

# unwrap common wrapper shapes
if isinstance(loaded, dict):
    if "state_dict" in loaded and isinstance(loaded["state_dict"], dict):
        sd = loaded["state_dict"]
    elif "model_state_dict" in loaded and isinstance(loaded["model_state_dict"], dict):
        sd = loaded["model_state_dict"]
    elif "checkpoint" in loaded and isinstance(loaded["checkpoint"], dict) and "state_dict" in loaded["checkpoint"]:
        sd = loaded["checkpoint"]["state_dict"]
    else:
        sd = loaded
elif isinstance(loaded, torch.nn.Module):
    print("Checkpoint is a full model object. Assigning directly to trainer.model")
    trainer.model = loaded
    sd = None
else:
    raise RuntimeError("Unsupported checkpoint format: " + str(type(loaded)))

if sd is None:
    print("Assigned full model from checkpoint to trainer.model. You're done.")
    raise SystemExit

print("state_dict entries:", len(sd))

# ensure trainer/model exists
if 'trainer' in globals() and hasattr(trainer, "model"):
    model_obj = trainer.model
    print("Using trainer.model as target.")
elif 'model' in globals():
    model_obj = model
    print("Using global 'model' as target.")
else:
    raise RuntimeError("No model object found in globals (need trainer.model or model).")

# collect parameter shapes from model
model_params = OrderedDict((n, tuple(p.shape)) for n, p in model_obj.named_parameters())
print("Target model parameter count:", len(model_params))

# helper: common suffix length (tokenized by '.')
def common_suffix_len(a, b):
    at = a.split('.')
    bt = b.split('.')
    i = 1
    cnt = 0
    while i <= len(at) and i <= len(bt):
        if at[-i] == bt[-i]:
            cnt += 1
            i += 1
        else:
            break
    return cnt

# prepare for matching
sd_keys = list(sd.keys())
used_sd = set()
mapping = {}  # model_key -> sd_key

# Try exact matches first
for mk in model_params:
    if mk in sd:
        # direct exact match, verify shape
        try:
            if tuple(sd[mk].shape) == model_params[mk]:
                mapping[mk] = mk
                used_sd.add(mk)
            else:
                # shapes differ -> skip exact
                pass
        except Exception:
            pass

# For remaining model keys, find best unused sd key by longest suffix match with same shape
for mk, mshape in model_params.items():
    if mk in mapping:
        continue
    best = None
    best_len = 0
    for sk in sd_keys:
        if sk in used_sd:
            continue
        # quick shape check
        try:
            if not torch.is_tensor(sd[sk]):
                continue
            if tuple(sd[sk].shape) != mshape:
                continue
        except Exception:
            continue
        # compute suffix match
        c = common_suffix_len(mk, sk)
        if c > best_len:
            best_len = c
            best = sk
    if best is not None and best_len >= 1:
        mapping[mk] = best
        used_sd.add(best)

mapped = len(mapping)
total_model = len(model_params)
print(f"Mapped {mapped}/{total_model} model params by suffix+shape matching.")

# show some example mappings
print("\nExamples of mappings (model_key <- checkpoint_key):")
for i,(mk,sk) in enumerate(list(mapping.items())[:30]):
    print(f" {i:03d}. {mk}  <-  {sk}")

# build remapped state_dict keyed by model param names (as expected by model.load_state_dict)
remapped_sd = OrderedDict()
for mk, sk in mapping.items():
    remapped_sd[mk] = sd[sk]

# Attempt safe load
print("\nAttempting load_state_dict(remapped_sd, strict=False)...")
try:
    missing, unexpected = model_obj.load_state_dict(remapped_sd, strict=False)
    # make lists friendly
    missing_l = list(missing) if hasattr(missing, '__iter__') else missing
    unexpected_l = list(unexpected) if hasattr(unexpected, '__iter__') else unexpected
    print("load_state_dict results -> missing keys:", len(missing_l), " unexpected keys:", len(unexpected_l))
    if len(missing_l):
        print(" sample missing:", missing_l[:12])
    if len(unexpected_l):
        print(" sample unexpected:", unexpected_l[:12])
except Exception:
    print("Exception during load_state_dict: trace below")
    traceback.print_exc()

# Save remapped state for debugging
out_remap = os.path.join(os.path.dirname(STATE_DICT_PATH), "remapped_state_dict_for_model.pth")
torch.save(remapped_sd, out_remap)
print("\nSaved remapped (model-keyed) state_dict to:", out_remap)

# Advice
if mapped / total_model > 0.85:
    print("\n✅ Good: >85% params mapped. Model should be mostly loaded. Classifier head (if any) may still be missing and needs re-init/fine-tune.")
elif mapped / total_model > 0.4:
    print("\n⚠️ Partial mapping (40-85%). You may have a config/arch mismatch for some blocks. Inspect the saved remapped_state_dict.")
else:
    print("\n❌ Poor mapping (<40%). Architecture/config mismatch likely. Paste the output above so I can inspect and advise next steps.")


NameError: name 'STATE_DICT_PATH' is not defined

In [ ]:
# ONE-CELL: fix missing gpt.lm_head.weight, save final remapped state_dict
import os, torch, traceback

# ensure model is present
if 'trainer' in globals() and hasattr(trainer, "model"):
    model_obj = trainer.model
else:
    model_obj = globals().get('model', None)

assert model_obj is not None, "No model object found (trainer.model or model)."

print("Model class:", type(model_obj))
# try to locate lm_head and embedding weights robustly (few HF variants exist)
lm_head = None
wte = None
try:
    # common locations
    if hasattr(model_obj, "gpt") and hasattr(model_obj.gpt, "lm_head"):
        lm_head = model_obj.gpt.lm_head
    elif hasattr(model_obj, "lm_head"):
        lm_head = model_obj.lm_head
    elif hasattr(model_obj, "gpt_head") and hasattr(model_obj.gpt_head, "lm_head"):
        lm_head = model_obj.gpt_head.lm_head
except Exception:
    pass

# find token embedding
try:
    if hasattr(model_obj, "gpt") and hasattr(model_obj.gpt, "transformer") and hasattr(model_obj.gpt.transformer, "wte"):
        wte = model_obj.gpt.transformer.wte
    elif hasattr(model_obj, "gpt") and hasattr(model_obj.gpt, "wte"):
        wte = model_obj.gpt.wte
    elif hasattr(model_obj, "transformer") and hasattr(model_obj.transformer, "wte"):
        wte = model_obj.transformer.wte
    elif hasattr(model_obj, "wte"):
        wte = model_obj.wte
except Exception:
    pass

print("Found lm_head:", bool(lm_head), " Found wte embedding:", bool(wte))

# Option A: tie lm_head weights to embedding (preferred)
if lm_head is not None and wte is not None:
    try:
        with torch.no_grad():
            # if shapes match, tie or copy
            if tuple(lm_head.weight.shape) == tuple(wte.weight.shape):
                print("Tying lm_head.weight to embedding (copy and set as same storage).")
                lm_head.weight = wte.weight
            else:
                print("Shapes differ; copying embedding values into lm_head (shape mismatch).")
                # copy as much as possible
                min_rows = min(lm_head.weight.shape[0], wte.weight.shape[0])
                min_cols = min(lm_head.weight.shape[1], wte.weight.shape[1])
                lm_head.weight.data[:min_rows, :min_cols] = wte.weight.data[:min_rows, :min_cols]
        fixed = True
    except Exception:
        print("Failed to tie/copy lm_head from embedding. Traceback:")
        traceback.print_exc()
        fixed = False
else:
    fixed = False

# Option B: fallback random init matching embedding std
if not fixed and lm_head is not None:
    try:
        print("Fallback: initializing lm_head with same std as embeddings (or normal if embeddings missing).")
        std = 0.02
        if wte is not None:
            try:
                std = float(wte.weight.std().cpu().item())
            except Exception:
                pass
        # reinit weights
        torch.nn.init.normal_(lm_head.weight, mean=0.0, std=std)
        fixed = True
    except Exception:
        print("Failed to init lm_head. Traceback:")
        traceback.print_exc()
        fixed = False

if not fixed:
    print("Could not fix lm_head automatically. You can manually set trainer.model.gpt.lm_head.weight = ... or assign trainer.model = loaded_full_model if checkpoint contained full model.")
else:
    # Save the fully matched state_dict (model-keyed) to drive next to original remapped file
    save_dir = os.path.dirname(STATE_DICT_PATH)
    out_path = os.path.join(save_dir, "model_state_dict_remapped_and_fixed.pth")
    # Use CPU-safe state_dict
    cpu_sd = {}
    for k,v in model_obj.state_dict().items():
        try:
            cpu_sd[k] = v.detach().cpu() if torch.is_tensor(v) else v
        except Exception:
            cpu_sd[k] = None
    torch.save(cpu_sd, out_path)
    print("Fixed lm_head and saved CPU state_dict to:", out_path)
    # Optionally overwrite trainer.model with the saved object (not necessary)


In [8]:
# 1) upgrade
!pip install -q -U "transformers" "datasets" "accelerate"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 17.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [9]:
# RUN THIS CELL BEFORE your quantize/eval cell
import os, torch
from torch.utils.data import DataLoader

# 1) Point to the remapped+fixed state_dict we saved earlier
REMAPPED_FIXED = "/content/drive/MyDrive/gpt2_agnews_checkpoints1/final_backup_auto/model_state_dict_remapped_and_fixed.pth"
if os.path.exists(REMAPPED_FIXED):
    print("Using remapped+fixed checkpoint:", REMAPPED_FIXED)
    STATE_DICT_PATH = REMAPPED_FIXED
else:
    print("Remapped+fixed file not found, falling back to original path:", STATE_DICT_PATH)

# 2) Ensure trainer exists and has tokenizer; if not, try to recover minimal pieces
if 'trainer' not in globals() or not hasattr(trainer, "model"):
    raise RuntimeError("trainer.model not found in globals. Make sure you ran the model/trainer creation cells.")

# 3) Provide an id2label mapping if your eval_fn expects it (AG News default)
if not hasattr(trainer, "id2label") and 'id2label' not in globals():
    id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
    print("No id2label found — using default AG News mapping:", id2label)
else:
    id2label = getattr(trainer, "id2label", globals().get("id2label"))

# 4) If trainer lacks get_eval_dataloader, create a small wrapper using HF dataset -> DataLoader
if not hasattr(trainer, "get_eval_dataloader"):
    print("trainer.get_eval_dataloader not found — creating a fallback that returns a DataLoader using trainer.data_collator")
    # expects DS is a HF Dataset (already tokenized) and trainer.data_collator exists
    def _fallback_get_eval_dataloader(ds, batch_size=8):
        coll = getattr(trainer, "data_collator", None)
        if coll is None:
            # fallback to simple collate (pads using tokenizer)
            tok = getattr(trainer, "tokenizer", None)
            if tok is None:
                raise RuntimeError("No tokenizer or data_collator available on trainer for fallback dataloader.")
            from transformers import DataCollatorWithPadding
            coll = DataCollatorWithPadding(tok)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=coll)
        return dl
    trainer.get_eval_dataloader = _fallback_get_eval_dataloader
else:
    print("trainer.get_eval_dataloader found — OK")

# 5) Minor robustness: ensure your TEST dataset variable is present (your big cell uses `test`)
if 'test' not in globals():
    raise RuntimeError("No `test` dataset found in globals. Run the tokenization/data prep cell first.")

print("Pre-flight checks OK. STATE_DICT_PATH =", STATE_DICT_PATH)


Using remapped+fixed checkpoint: /content/drive/MyDrive/gpt2_agnews_checkpoints1/final_backup_auto/model_state_dict_remapped_and_fixed.pth
No id2label found — using default AG News mapping: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
trainer.get_eval_dataloader found — OK
Pre-flight checks OK. STATE_DICT_PATH = /content/drive/MyDrive/gpt2_agnews_checkpoints1/final_backup_auto/model_state_dict_remapped_and_fixed.pth


In [10]:
# SAFE: One-cell quantize + dequant-for-eval (NO HF HUB calls anywhere)
import os, time, json, traceback, torch
from copy import deepcopy
from collections import OrderedDict
from torch.utils.data import DataLoader

# -------- CONFIG (edit only if you want) ----------
STATE_DICT_PATH = "/content/drive/MyDrive/gpt2_agnews_checkpoints1/final_backup_auto/model_state_dict_remapped_and_fixed.pth"
OUT_FOLDER = os.path.join("/content/drive/MyDrive", "AGNEWS_QUANTIZED_SAFE_" + time.strftime("%Y%m%d_%H%M%S"))
os.makedirs(OUT_FOLDER, exist_ok=True)
MIN_ELEMENTS_TO_QUANT = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --------------------------------------------------

print("STATE_DICT_PATH:", STATE_DICT_PATH)
assert os.path.exists(STATE_DICT_PATH), "State dict not found at the path above."

# load the checkpoint (local only)
loaded = torch.load(STATE_DICT_PATH, map_location="cpu")
if isinstance(loaded, dict) and "state_dict" in loaded and isinstance(loaded["state_dict"], dict):
    sd = loaded["state_dict"]
elif isinstance(loaded, dict):
    sd = loaded
elif isinstance(loaded, torch.nn.Module):
    # full model saved — assign directly
    trainer.model = loaded
    sd = None
else:
    raise RuntimeError("Unsupported checkpoint format: " + str(type(loaded)))

# Ensure trainer.model exists
if 'trainer' not in globals() or not hasattr(trainer, "model"):
    raise RuntimeError("trainer.model not found in globals. Load/instantiate trainer.model before running this cell.")

# Use a deepcopy of trainer.model to avoid constructor/config pitfalls
print("Cloning trainer.model (deepcopy) — this avoids constructing from config or touching the hub.")
new_model = deepcopy(trainer.model)
new_model.eval()

# If we do have a state_dict, load it into new_model safely (strip common wrappers)
if sd is not None:
    clean_sd = {}
    for k, v in sd.items():
        nk = k
        if nk.startswith("module."): nk = nk[len("module."):]
        if nk.startswith("model."):  nk = nk[len("model."):]
        clean_sd[nk] = v
    missing, unexpected = new_model.load_state_dict(clean_sd, strict=False)
    print("Loaded state_dict into model: missing:", len(missing), " unexpected:", len(unexpected))
    if missing:
        print(" Examples missing:", list(missing)[:12])
    if unexpected:
        print(" Examples unexpected:", list(unexpected)[:12])

# ---- quantization helpers (symmetric int8 linear) ----
def quantize_tensor_sym_int8(t: torch.Tensor):
    t = t.detach().cpu()
    maxv = float(t.abs().max().item())
    scale = float(maxv / 127.0) if maxv != 0.0 else 1.0
    q = torch.clamp((t / scale).round(), -127, 127).to(torch.int8)
    return q, float(scale)

def dequantize_tensor_sym_int8(q: torch.Tensor, scale: float, device=None):
    deq = q.to(torch.float32) * float(scale)
    return deq.to(device) if device is not None else deq

# ---- do quantization (skip biases / tiny / layernorm-like) ----
quantized_store = {}
dequant_for_eval = {}

print("Quantizing parameters...")
for name, param in new_model.named_parameters():
    if name.endswith(".bias"):
        continue
    if param.numel() < MIN_ELEMENTS_TO_QUANT:
        continue
    low = name.lower()
    if ("layernorm" in low or "ln_" in low or (("norm" in low) and param.ndim == 1)):
        continue
    try:
        q, scale = quantize_tensor_sym_int8(param.data)
        quantized_store[name] = {"q": q, "scale": scale, "orig_shape": list(param.shape)}
        deq = dequantize_tensor_sym_int8(q, scale, device=None).view(param.shape).to(torch.float32)
        dequant_for_eval[name] = deq
    except Exception as e:
        print("Skipped quantizing", name, "due to:", e)

print(f"Quantized {len(quantized_store)} tensors. Preparing to save quantized_state.pth in {OUT_FOLDER}")

# save quantized state (int8 tensors + scales)
quant_save_path = os.path.join(OUT_FOLDER, "quantized_state.pth")
torch.save(quantized_store, quant_save_path)
print("Saved quantized_state.pth ->", quant_save_path)

# metadata
total_q_elems = sum(int(torch.numel(v["q"])) for v in quantized_store.values())
meta = {"num_quantized_tensors": len(quantized_store), "quant_elems": total_q_elems}
with open(os.path.join(OUT_FOLDER, "quant_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

# ---- prepare eval model by copying then assigning dequantized params ----
eval_model = deepcopy(new_model)  # clone model with same structure
# copy original state then overwrite quantized params with dequantized floats
try:
    eval_model.load_state_dict(new_model.state_dict(), strict=False)
except Exception as e:
    print("Warning while copying state dict to eval_model:", e)

for name, p in eval_model.named_parameters():
    if name in dequant_for_eval:
        with torch.no_grad():
            p.data = dequant_for_eval[name].to(p.device)

eval_model.to(DEVICE)
eval_model.eval()

# ---- minimal evaluator (safe, local; will not call HF hub) ----
# requires `test` dataset to be a tokenized HF Dataset (dict-like with tensors)
if 'test' not in globals():
    raise RuntimeError("No `test` dataset present. Run tokenization cell before this.")

# fallback dataloader if trainer.get_eval_dataloader missing
if hasattr(trainer, "get_eval_dataloader"):
    get_eval_dl = trainer.get_eval_dataloader
else:
    coll = getattr(trainer, "data_collator", None)
    tok = getattr(trainer, "tokenizer", None)
    if coll is None:
        if tok is None:
            raise RuntimeError("No data_collator or tokenizer available to make eval dataloader.")
        from transformers import DataCollatorWithPadding
        coll = DataCollatorWithPadding(tok)
    def _get_eval_dl(ds, batch_size=8):
        return DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=coll)
    get_eval_dl = _get_eval_dl

mini = type("Mini", (), {})()
mini.model = eval_model
mini.get_eval_dataloader = get_eval_dl
mini.tokenizer = getattr(trainer, "tokenizer", None)
mini.id2label = getattr(trainer, "id2label", None) or {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# the minimal evaluation function (fast, safe)
def minimal_eval(mini_trainer, dataset, max_samples=None):
    dl = mini_trainer.get_eval_dataloader(dataset)
    model = mini_trainer.model
    device = next(model.parameters()).device
    correct = 0
    total = 0
    with torch.no_grad():
        for i, batch in enumerate(dl):
            b = {k: v.to(device) for k,v in batch.items() if isinstance(v, torch.Tensor)}
            out = model(input_ids=b.get("input_ids"), attention_mask=b.get("attention_mask"))
            logits = out.get("logits") if isinstance(out, dict) else (out[0] if isinstance(out, (list,tuple)) else None)
            if logits is None:
                raise RuntimeError("model forward did not return logits as expected.")
            preds = logits.argmax(dim=-1).detach().cpu().tolist()
            labels = b.get("labels")
            if labels is None:
                break
            labels = labels.detach().cpu().tolist()
            for p,l in zip(preds, labels):
                correct += int(p==l)
                total += 1
            if max_samples and total >= max_samples:
                break
    return {"accuracy": correct/total if total>0 else None, "total": total}

# run evaluation
print("Running safe minimal evaluation on quantized (dequant) model...")
metrics = minimal_eval(mini, test, max_samples=None)
print("Evaluation metrics:", metrics)

# save summary + artifacts
with open(os.path.join(OUT_FOLDER, "metrics_quant_safe.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved artifacts to:", OUT_FOLDER)


STATE_DICT_PATH: /content/drive/MyDrive/gpt2_agnews_checkpoints1/final_backup_auto/model_state_dict_remapped_and_fixed.pth
Cloning trainer.model (deepcopy) — this avoids constructing from config or touching the hub.
Loaded state_dict into model: missing: 0  unexpected: 0
Quantizing parameters...
Quantized 51 tensors. Preparing to save quantized_state.pth in /content/drive/MyDrive/AGNEWS_QUANTIZED_SAFE_20251007_134149


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Saved quantized_state.pth -> /content/drive/MyDrive/AGNEWS_QUANTIZED_SAFE_20251007_134149/quantized_state.pth
Running safe minimal evaluation on quantized (dequant) model...
Evaluation metrics: {'accuracy': 0.9411842105263157, 'total': 7600}
Saved artifacts to: /content/drive/MyDrive/AGNEWS_QUANTIZED_SAFE_20251007_134149


In [11]:
# ONE CELL: Detailed metrics, confusion matrix, latency, save artifacts
import os, time, json, torch, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report, accuracy_score

# ---------- CONFIG ----------
OUT_FOLDER = OUT_FOLDER  # uses existing OUT_FOLDER from your quantize cell
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_SAMPLES = None   # set to int to limit (e.g., 2000), or None to run full test set
BATCH_SIZE_FOR_LATENCY = 8
# ------------------------

assert 'mini' in globals(), "mini wrapper not found. Ensure quantize/eval cell created `mini`."
assert 'test' in globals(), "test dataset not found. Run tokenization cell if needed."

model = mini.model
model.eval()
model.to(DEVICE)

# Get dataloader (use mini.get_eval_dataloader if provided)
dl = mini.get_eval_dataloader(test)
# If HF Dataset -> DataLoader yields dict with tensors; if not, adapt as needed

# helper to iterate and collect preds/labels
all_preds = []
all_labels = []
sample_count = 0

with torch.no_grad():
    for batch in dl:
        # convert tensors to device
        b = {}
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                b[k] = v.to(DEVICE)
        outputs = model(input_ids=b.get("input_ids"), attention_mask=b.get("attention_mask"))
        logits = outputs.get("logits") if isinstance(outputs, dict) else (outputs[0] if isinstance(outputs, (list,tuple)) else None)
        if logits is None:
            raise RuntimeError("Model forward did not return logits in expected format.")
        preds = logits.argmax(dim=-1).detach().cpu().tolist()
        labels = b.get("labels")
        if labels is None:
            break
        labels = labels.detach().cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels)
        sample_count += len(labels)
        if MAX_SAMPLES and sample_count >= MAX_SAMPLES:
            break

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
print("Collected predictions for", len(all_preds), "samples.")

# compute metrics
acc = float(accuracy_score(all_labels, all_preds))
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(all_labels, all_preds, average='micro', zero_division=0)
per_class = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=sorted(list(set(all_labels))), zero_division=0)
report_text = classification_report(all_labels, all_preds, digits=4, zero_division=0)
cm = confusion_matrix(all_labels, all_preds)

# save classification report
cr_path = os.path.join(OUT_FOLDER, "classification_report_quant.txt")
with open(cr_path, "w") as f:
    f.write(report_text)

# save confusion matrix plot
plt.figure(figsize=(6,6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion matrix (quantized model)")
plt.colorbar()
classes = [mini.id2label[i] if (hasattr(mini, "id2label") and i in mini.id2label) else str(i) for i in range(cm.shape[0])]
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation=45, ha="right")
plt.yticks(tick_marks, classes)
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
cm_path = os.path.join(OUT_FOLDER, "confusion_matrix_quant.png")
plt.savefig(cm_path)
plt.close()

# latency measurement (simple)
# Warmup
print("Warming up GPU/CPU (5 batches)...")
model.eval()
with torch.no_grad():
    it = iter(dl)
    for _ in range(5):
        try:
            batch = next(it)
        except StopIteration:
            break
        b = {k: v.to(DEVICE) for k,v in batch.items() if isinstance(v, torch.Tensor)}
        _ = model(input_ids=b.get("input_ids"), attention_mask=b.get("attention_mask"))

# Timed runs
print("Measuring latency on", BATCH_SIZE_FOR_LATENCY, "batch size for 50 batches (or until data exhausted)...")
times = []
with torch.no_grad():
    it = iter(dl)
    for i in range(50):
        try:
            batch = next(it)
        except StopIteration:
            break
        b = {k: v.to(DEVICE) for k,v in batch.items() if isinstance(v, torch.Tensor)}
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t0 = time.time()
        _ = model(input_ids=b.get("input_ids"), attention_mask=b.get("attention_mask"))
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t1 = time.time()
        times.append(t1 - t0)
if len(times) > 0:
    avg_batch_time = float(np.mean(times))
    avg_time_per_sample = avg_batch_time / float(BATCH_SIZE_FOR_LATENCY)
else:
    avg_batch_time = None
    avg_time_per_sample = None

# compute size comparisons
def sizeof_state_dict(sd):
    return sum(int(v.numel() * 4) for v in sd.values() if torch.is_tensor(v))  # approx bytes assuming float32

orig_sd = trainer.model.state_dict()
orig_bytes = sizeof_state_dict(orig_sd)
quant_bytes = 0
try:
    # if you saved quantized_state.pth earlier load it
    qpath = os.path.join(OUT_FOLDER, "quantized_state.pth")
    if os.path.exists(qpath):
        qdict = torch.load(qpath, map_location="cpu")
        quant_bytes = sum(int(torch.numel(v["q"])) for v in qdict.values())
    else:
        # fallback: use quantized_store variable if in scope
        if 'quantized_store' in globals():
            quant_bytes = sum(int(torch.numel(quantized_store[k]["q"])) for k in quantized_store)
except Exception:
    quant_bytes = None

summary = {
    "accuracy": acc,
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "precision_micro": float(precision_micro),
    "recall_micro": float(recall_micro),
    "f1_micro": float(f1_micro),
    "per_class_precision": per_class[0].tolist() if isinstance(per_class[0], np.ndarray) else list(per_class[0]),
    "per_class_recall": per_class[1].tolist() if isinstance(per_class[1], np.ndarray) else list(per_class[1]),
    "per_class_f1": per_class[2].tolist() if isinstance(per_class[2], np.ndarray) else list(per_class[2]),
    "confusion_matrix_shape": list(cm.shape),
    "num_samples_evaluated": int(len(all_labels)),
    "avg_batch_time_s": avg_batch_time,
    "avg_time_per_sample_s": avg_time_per_sample,
    "orig_model_bytes_est": int(orig_bytes),
    "quant_storage_bytes": int(quant_bytes) if quant_bytes is not None else None,
}

# save summary json and classification report already saved
with open(os.path.join(OUT_FOLDER, "quant_eval_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Saved classification report:", cr_path)
print("Saved confusion matrix image:", cm_path)
print("Saved eval summary JSON:", os.path.join(OUT_FOLDER, "quant_eval_summary.json"))
print("\nSummary metrics:")
print(json.dumps(summary, indent=2))


Collected predictions for 7600 samples.
Warming up GPU/CPU (5 batches)...
Measuring latency on 8 batch size for 50 batches (or until data exhausted)...
Saved classification report: /content/drive/MyDrive/AGNEWS_QUANTIZED_SAFE_20251007_134149/classification_report_quant.txt
Saved confusion matrix image: /content/drive/MyDrive/AGNEWS_QUANTIZED_SAFE_20251007_134149/confusion_matrix_quant.png
Saved eval summary JSON: /content/drive/MyDrive/AGNEWS_QUANTIZED_SAFE_20251007_134149/quant_eval_summary.json

Summary metrics:
{
  "accuracy": 0.9411842105263157,
  "precision_macro": 0.941118090952686,
  "recall_macro": 0.9411842105263158,
  "f1_macro": 0.9410689122977606,
  "precision_micro": 0.9411842105263157,
  "recall_micro": 0.9411842105263157,
  "f1_micro": 0.9411842105263157,
  "per_class_precision": [
    0.95020964360587,
    0.9817136886102403,
    0.9234111895708854,
    0.909137842023748
  ],
  "per_class_recall": [
    0.9542105263157895,
    0.9889473684210527,
    0.8947368421052632,